# EfficientNet — Fine-tuning JUSTO (comparação controlada com o YOLO11s)

**Objetivo:** isolar a pergunta "foi a arquitetura ou a receita?". Esta versão reproduz
EXATAMENTE as condições do `finetune_yolo.ipynb`, mudando só o backbone (EfficientNet-B0):

| Condição | YOLO11s-cls | EfficientNet (aqui) |
|---|---|---|
| fotos | as 57 reais | **as mesmas 57** |
| crop | OpenCV (app) | **o mesmo** |
| split | seed=42, val=12 | **idêntico (mesma lógica do FT-1)** |
| augmentation | HSV forte + flips + erasing | **equivalente** |
| freeze | último bloco + cabeça | **block7 + top + cabeça (BN congelada)** |
| dados extras | nenhum | **nenhum (sem mix do Roboflow)** |

A run antiga (64%) NÃO era comparável: usava `lr=1e-5` (100× menor), misturava 60% de
dados do Roboflow e media num holdout de ~100 fotos. Aqui o val é o MESMO de 12 fotos do
YOLO — o número sai lado a lado.

➡️ Rode EFF-0 → EFF-4. Self-contained (sobrevive a restart).

In [ ]:
# EFF-0 — BOOTSTRAP (self-contained)
!pip -q install huggingface_hub

import os, glob, random, shutil, tempfile, pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download

tf.keras.utils.set_random_seed(42)
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Sem GPU! Ambiente de execucao -> Alterar tipo -> T4'

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive'

CLASS_NAMES = ['Broken soybeans', 'Immature soybeans', 'Intact soybeans',
               'Skin-damaged soybeans', 'Spotted soybeans']
SHORT = {'Broken soybeans':'broken', 'Immature soybeans':'immature', 'Intact soybeans':'intact',
         'Skin-damaged soybeans':'skin-damaged', 'Spotted soybeans':'spotted'}
short_names = [SHORT[c] for c in CLASS_NAMES]
idx_of = {n: i for i, n in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

def crop_single_grain(arr):
    h, w = arr.shape[:2]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr
    largest = max(contours, key=cv2.contourArea)
    if not (0.02 < cv2.contourArea(largest) / (h * w) < 0.95):
        return arr
    x, y, bw, bh = cv2.boundingRect(largest)
    pad = max(15, int(min(bw, bh) * 0.12))
    return arr[max(0, y-pad):min(h, y+bh+pad), max(0, x-pad):min(w, x+bw+pad)]

# Modelo base = EfficientNet treinado no Roboflow (mesmo papel do soja_yolo11s_best.pt)
BASE_KERAS = f'{SAVE_DIR}/soja_model_final.keras'
assert os.path.exists(BASE_KERAS), f'NAO achei {BASE_KERAS} no Drive (gerado pelo train.ipynb).'
model = keras.models.load_model(BASE_KERAS)
model.compile(loss='categorical_crossentropy', metrics=['accuracy'])
print('OK - EfficientNet base carregado de', BASE_KERAS)

HF_TOKEN  = getpass('Cole seu HF token (read): ')
CORR_REPO = 'Guguinhaxd/soja-correction'
api = HfApi(token=HF_TOKEN)
print('OK - bootstrap pronto.')

In [ ]:
# EFF-1 — Mesmo split do FT-1 do YOLO (seed=42 -> val identico de 12 fotos)
# Lógica BYTE-A-BYTE igual ao finetune_yolo.ipynb: mesmo random.Random(42), mesma
# ordem de classes, mesmo sorted()+shuffle, mesmo round(0.2*n). Garante que o val
# aqui são EXATAMENTE as 12 mesmas fotos que o YOLO avaliou.
REAL_DST = '/content/soja_real'
if os.path.exists(REAL_DST):
    shutil.rmtree(REAL_DST)
rng = random.Random(42)
IMG_EXTS_SET = {'.jpg', '.jpeg', '.png'}
tmpd = tempfile.mkdtemp(prefix='efft_')

all_files = list(api.list_repo_files(CORR_REPO, repo_type='dataset'))
summary = {}
for cls in CLASS_NAMES:
    fs = sorted(f for f in all_files
                if f.startswith(f'{cls}/') and pathlib.Path(f).suffix.lower() in IMG_EXTS_SET)
    rng.shuffle(fs)
    n_val = max(1, round(len(fs) * 0.2)) if len(fs) >= 2 else 0
    val_set = set(fs[:n_val])
    for f in fs:
        local = hf_hub_download(CORR_REPO, f, repo_type='dataset', token=HF_TOKEN, local_dir=tmpd)
        arr  = np.array(Image.open(local).convert('RGB'))
        crop = crop_single_grain(arr)
        split  = 'val' if f in val_set else 'train'
        outdir = os.path.join(REAL_DST, split, cls)
        os.makedirs(outdir, exist_ok=True)
        Image.fromarray(crop).save(os.path.join(outdir, pathlib.Path(f).stem + '.jpg'), quality=95)
    summary[SHORT[cls]] = (len(fs) - n_val, n_val)

print('classe -> (train, val)')
for k, v in summary.items():
    print(f'  {k:>13}: {v}')
print('\nval esperado p/ bater com o YOLO: broken3 immature2 intact4 skin2 spotted1 = 12')

In [ ]:
# EFF-2 — Pipelines tf.data + augmentation FORTE equivalente ao YOLO + baseline
IMG_SIZE = (224, 224)
BATCH    = 8                 # mesmo batch do YOLO
AUTOTUNE = tf.data.AUTOTUNE

def list_split(split):
    paths, labels = [], []
    for cls in CLASS_NAMES:
        for p in sorted(glob.glob(os.path.join(REAL_DST, split, cls, '*.jpg'))):
            paths.append(p); labels.append(idx_of[cls])
    return paths, labels

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)               # [0,255]: EfficientNet faz o preprocess interno
    return img, tf.one_hot(label, NUM_CLASSES)

def random_erasing(x, p=0.4):                    # ~ erasing=0.4 do YOLO
    def erase():
        H = W = 224
        target = tf.random.uniform([], 0.02, 0.2) * (H * W)
        ar = tf.random.uniform([], 0.3, 3.3)
        eh = tf.minimum(tf.cast(tf.round(tf.sqrt(target * ar)), tf.int32), H)
        ew = tf.minimum(tf.cast(tf.round(tf.sqrt(target / ar)), tf.int32), W)
        top  = tf.random.uniform([], 0, H - eh + 1, dtype=tf.int32)
        left = tf.random.uniform([], 0, W - ew + 1, dtype=tf.int32)
        rows = tf.range(H)[:, None]; cols = tf.range(W)[None, :]
        mask = tf.cast((rows >= top) & (rows < top + eh) &
                       (cols >= left) & (cols < left + ew), x.dtype)[..., None]
        noise = tf.random.uniform([H, W, 3], 0., 1., dtype=x.dtype)
        return x * (1 - mask) + noise * mask
    return tf.cond(tf.random.uniform([]) < p, erase, lambda: x)

def color_aug(img, label):                       # ~ HSV + flips do YOLO
    x = img / 255.0
    x = tf.image.random_flip_left_right(x)       # fliplr=0.5
    x = tf.image.random_flip_up_down(x)          # flipud=0.5
    x = tf.image.random_saturation(x, 0.3, 1.7)  # ~ hsv_s=0.7
    x = tf.image.random_hue(x, 0.02)             #   hsv_h=0.02
    x = tf.image.random_brightness(x, 0.4)       # ~ hsv_v=0.6
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.clip_by_value(x, 0., 1.)
    x = random_erasing(x, p=0.4)
    return x * 255.0, label

geo_aug = keras.Sequential([                     # geometria do YOLO
    layers.RandomRotation(15 / 360.),            # degrees=15
    layers.RandomZoom(0.4),                      # ~ scale=0.5
    layers.RandomTranslation(0.1, 0.1),          # translate=0.1
], name='geo_aug')

tr_paths, tr_labels = list_split('train')
va_paths, va_labels = list_split('val')

train_ds = (tf.data.Dataset.from_tensor_slices((tr_paths, tr_labels))
            .shuffle(len(tr_paths), seed=42, reshuffle_each_iteration=True)
            .map(load_image, num_parallel_calls=AUTOTUNE)
            .map(color_aug,  num_parallel_calls=AUTOTUNE)
            .batch(BATCH)
            .map(lambda x, y: (geo_aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))

def eval_ds(paths, labels):
    return (tf.data.Dataset.from_tensor_slices((paths, labels))
            .map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE))

train_eval = eval_ds(tr_paths, tr_labels)
val_eval   = eval_ds(va_paths, va_labels)

_, base_tr = model.evaluate(train_eval, verbose=0)
_, base_va = model.evaluate(val_eval,   verbose=0)
print(f'BASELINE EfficientNet no dominio real: train {base_tr:.1%} | val {base_va:.1%}')
print('(YOLO baseline: train 2.2% | val 8.3%. Se aqui der parecido, confirma que o')
print(' soja_model_final.keras e a base pura do Roboflow -> comparacao valida.)')

In [ ]:
# EFF-3 — Descongela último bloco + cabeça (equivale ao freeze=9 do YOLO) e treina
base_model = next((l for l in model.layers if isinstance(l, keras.Model)), None)
if base_model is not None:
    base_model.trainable = True
    for layer in base_model.layers:
        last_block = layer.name.startswith('block7') or layer.name.startswith('top')
        layer.trainable = last_block and not isinstance(layer, layers.BatchNormalization)
    n_tr = sum(1 for l in base_model.layers if l.trainable)
    print(f'Treinaveis na base: {n_tr}/{len(base_model.layers)} (block7 + top, BN congelada)')
else:
    print('Base aninhada nao encontrada — treinando so a cabeca.')

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),       # comparavel ao LR efetivo do YOLO (nao 1e-5!)
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True,
                                  monitor='val_accuracy', verbose=1),   # mesmo patience do YOLO
    keras.callbacks.ReduceLROnPlateau(patience=8, factor=0.5, monitor='val_loss',
                                      min_lr=1e-7, verbose=1),
]
history = model.fit(train_ds, validation_data=val_eval, epochs=80, callbacks=callbacks)
print('\nFine-tuning concluido.')

In [ ]:
# EFF-4 — Antes x Depois + matriz de confusao + comparativo final
from sklearn.metrics import classification_report, confusion_matrix

_, new_tr = model.evaluate(train_eval, verbose=0)
_, new_va = model.evaluate(val_eval,   verbose=0)
y_pred = np.argmax(model.predict(val_eval, verbose=0), axis=1)
y_true = np.array(va_labels)

print('=== EfficientNet (aug FORTE + freeze parcial) — dominio real ===')
print(f'  train: {base_tr:.1%}  ->  {new_tr:.1%}')
print(f'  val:   {base_va:.1%}  ->  {new_va:.1%}')
gap = new_tr - new_va
print(f'  gap depois: {gap:.1%}  ', end='')
print('(overfitting)' if gap > 0.2 else '(saudavel)')

print('\n--- COMPARATIVO (mesmo val de 12 fotos) ---')
print(f'  EfficientNet aug forte (este):       val {new_va:.1%}  | gap {gap:.1%}')
print(f'  EfficientNet aug branda + mix (antigo): val 64.0%    | gap 25.0%')
print(f'  YOLO11s-cls aug forte:               val 91.7%    | gap 3.9%')

print('\n=== Relatorio por classe (val real) ===')
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES),
                            target_names=short_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=short_names, yticklabels=short_names,
            cmap='Purples', linewidths=0.5)
plt.xlabel('Predito'); plt.ylabel('Real')
plt.title(f'EfficientNet aug forte — val {new_va:.0%}')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# EFF-5 (OPCIONAL) — Salva o EfficientNet justo no Drive (so se quiser guardar)
dest = f'{SAVE_DIR}/soja_eff_fair_finetuned.keras'
model.save(dest)
print('Salvo em', dest)
print(f'val real: {new_va:.1%}  (compare com YOLO 91.7%)')